# Triton Kernel Profiling on AMD GPUs

## Get the System Hardware Info

In [1]:
!rocminfo

ROCk module version 6.16.6 is loaded
HSA System Attributes    
Runtime Version:         1.18
Runtime Ext Version:     1.11
System Timestamp Freq.:  1000.000000MHz
Sig. Max Wait Duration:  18446744073709551615 (0xFFFFFFFFFFFFFFFF) (timestamp count)
Machine Model:           LARGE                              
System Endianness:       LITTLE                             
Mwaitx:                  DISABLED
XNACK enabled:           NO
DMAbuf Support:          YES
VMM Support:             YES

HSA Agents               
*******                  
Agent 1                  
*******                  
  Name:                    Intel(R) Xeon(R) Platinum 8470     
  Uuid:                    CPU-XX                             
  Marketing Name:          Intel(R) Xeon(R) Platinum 8470     
  Vendor Name:             CPU                                
  Feature:                 None specified                     
  Profile:                 FULL_PROFILE                       
  Float Round Mode:        

## Check profiling environment

In [2]:
!rocprof-compute --version

----------------------------------------
rocprofiler-compute version: 3.2.3 (release)
Git revision:     1d722c14
----------------------------------------


## Non-Fusion MatMulBias Kernel

In [3]:
%%writefile  nonfusion_matmulbias.py
import argparse
import torch

import triton
import triton.language as tl
import triton.profiler as proton

def _matmul_launch_metadata(grid, kernel, args):
    ret = {}
    M, N, K, WS = args["M"], args["N"], args["K"], args.get("WARP_SPECIALIZE", False)
    BM, BN, BK = args["BLOCK_SIZE_M"], args["BLOCK_SIZE_N"], args["BLOCK_SIZE_K"]
    ws_str = "_ws" if WS else ""
    ret["name"] = f"{kernel.name}{ws_str} [M={M}, N={N}, K={K}] [BM={BM}, BN={BN} BK={BK}]"
    if "output_ptr" in args:
        bytes_per_elem = args["output_ptr"].element_size()
    else:
        bytes_per_elem = 2
    ret[f"flops{bytes_per_elem * 8}"] = 2. * M * N * K
    ret["bytes"] = bytes_per_elem * (M * K + N * K + M * N)
    return ret

def _bias_launch_metadata(grid, kernel, args):
    ret = {}
    M, N, WS = args["M"], args["N"], args.get("WARP_SPECIALIZE", False)
    BM, BN = args["BLOCK_SIZE_M"], args["BLOCK_SIZE_N"]
    ws_str = "_ws" if WS else ""
    ret["name"] = f"{kernel.name}{ws_str} [M={M}, N={N}] [BM={BM}, BN={BN}]"
    if "output_ptr" in args:
        bytes_per_elem = args["output_ptr"].element_size()
    else:
        bytes_per_elem = 2
    ret[f"flops{bytes_per_elem * 8}"] = 2. * M * N
    ret["bytes"] = bytes_per_elem * (M + N + M * N)
    return ret


def matmul_autotune_config(pre_hook=None):
    return [
        triton.Config({'BLOCK_SIZE_M': BM, 'BLOCK_SIZE_N': BN, "BLOCK_SIZE_K": BK, "GROUP_SIZE_M": 8}, num_stages=s,
            num_warps=w, pre_hook=pre_hook)
        for BM in [128]
        for BN in [128, 256]
        for BK in [64, 128]
        for s in ([3, 4, 5])
        for w in [4, 8]
    ]

def bias_autotune_config(pre_hook=None):
    return [
        triton.Config({'BLOCK_SIZE_M': BM, 'BLOCK_SIZE_N': BN, "GROUP_SIZE_M": 8}, num_stages=s,
            num_warps=w, pre_hook=pre_hook)
        for BM in [128]
        for BN in [128, 256]
        for s in ([3, 4, 5])
        for w in [4, 8]
    ]

# MatMul kernel
@triton.autotune(
    configs=matmul_autotune_config(),
    key=['M', 'N', 'K'],
)
@triton.jit(launch_metadata=_matmul_launch_metadata)
def matmul_kernel(
        a_ptr, b_ptr, c_ptr,
        M, N, K,
        stride_am, stride_ak,
        stride_bk, stride_bn,
        stride_cm, stride_cn,
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,
        GROUP_SIZE_M: tl.constexpr
):
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    tl.assume(pid_m >= 0)
    tl.assume(pid_n >= 0)
    tl.assume(stride_am > 0)
    tl.assume(stride_ak > 0)
    tl.assume(stride_bn > 0)
    tl.assume(stride_bk > 0)
    tl.assume(stride_cm > 0)
    tl.assume(stride_cn > 0)

    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        accumulator = tl.dot(a, b, accumulator)
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk
    c = accumulator.to(tl.float16)

    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)

# Bias kernel
@triton.autotune(
    configs=bias_autotune_config(),
    key=['M', 'K'],
)
@triton.jit(launch_metadata=_bias_launch_metadata)
def bias_kernel(
        a_ptr,
        bias_ptr,
        output_ptr,
        M, N,
        stride_m, stride_n,
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, GROUP_SIZE_M: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    offs_m = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_n = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    a_ptrs = a_ptr + (offs_m[:, None] * stride_m + offs_n[None, :] * stride_n)
    
    a = tl.load(a_ptrs)
    
    bias = tl.load(bias_ptr + offs_n, mask=offs_n < N, other=0.0)

    output = a + bias[None, :]
        
    offs_m = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_n = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    output_ptrs = output_ptr + stride_m * offs_m[:, None] + stride_n * offs_n[None, :]
    output_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(output_ptrs, output, mask=output_mask)


# MatMulBias kernel wrapper function
def matmulbias(a: torch.Tensor, b: torch.Tensor, bias: torch.Tensor):
    assert a.shape[1] == b.shape[0], "Incompatible dimensions"
    assert a.is_contiguous(), "Matrix A must be contiguous"
    M, K = a.shape
    K, N = b.shape
    assert bias.shape[0] == N, "BIAS has incompatible dimensions"
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    matmul_grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']), )
    matmul_kernel[matmul_grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
    )

    o = torch.empty((M, N), device=c.device, dtype=torch.float16)
    bias_grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']), )
    bias_kernel[bias_grid](
        c, bias, o,
        M, N,
        c.stride(0), c.stride(1),
    )
    
    return o


def get_rtol():
    target = triton.runtime.driver.active.get_current_target()
    if target.backend == "hip":
        if target.arch == "gfx90a":
            return 1e-2
        elif target.arch == "gfx942":
            return 1e-2
    else:
        return 0

def show_profile(profile_name):
    import triton.profiler.viewer as proton_viewer
    metric_names = ["time/ms", "tflop16/s"]
    file_name = f"{profile_name}.hatchet"
    tree, metrics = proton_viewer.parse(metric_names, file_name)

    print(f"Proton profile results for {profile_name}")
    proton_viewer.print_tree(tree, metrics)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--profile", action="store_true")
    parser.add_argument("--verify", action="store_true")
    args = parser.parse_args()

    # Test matrices
    torch.manual_seed(0)
    M = 1024
    N = 1024
    a = torch.randn((M, N), device='cuda', dtype=torch.float16)
    b = torch.randn((N, M), device='cuda', dtype=torch.float16)
    bias = torch.randn(N, device='cuda', dtype=torch.float16)
    # bias = torch.zeros(N, device='cuda', dtype=torch.float16)

    if args.profile:
        print("Profiling the nonfusion matmulbias kernel ...")
        proton.start("nonfusion_matmulbias", hook="triton")
    else:
        print("Running the nonfusion matmulbias kernel ...")
    nonfusion_triton_output = matmulbias(a, b, bias)
    if args.profile:
        proton.finalize()
    print(f"nonfusion_triton_output: {nonfusion_triton_output}")

    if args.profile:
        show_profile("nonfusion_matmulbias")

    if args.verify:
        torch_output = torch.addmm(bias, a, b)
        print(f"torch_output: {torch_output}")
        print("Verifying triton results with torch ...")
        triton.testing.assert_close(nonfusion_triton_output, torch_output, atol=1e-2, rtol=get_rtol())
        print("OK")


Overwriting nonfusion_matmulbias.py


### Pre-Build the Non-Fusion MatMulBias Kernel

In [4]:
!python nonfusion_matmulbias.py --verify

Running the nonfusion matmulbias kernel ...
nonfusion_triton_output: tensor([[  2.7480,  43.6250,  30.9531,  ...,  26.2500,   8.8203,   6.4609],
        [  7.1562, -21.3906,  17.7969,  ..., -16.8125,  29.6562, -48.3125],
        [ 13.8125,  -6.4609,   0.5146,  ...,  48.0000,  25.1562, -27.5156],
        ...,
        [-54.4375,  -1.4375,  -0.4214,  ...,   5.4297,  17.2812, -39.0625],
        [  9.6094, -52.2500,  43.0000,  ..., -45.8438, -49.3125,  32.6875],
        [-12.8516,   9.5391,  53.3750,  ...,  -9.3438,   7.6797,  54.1250]],
       device='cuda:0', dtype=torch.float16)
torch_output: tensor([[  2.7480,  43.6562,  30.9531,  ...,  26.2500,   8.8203,   6.4609],
        [  7.1602, -21.3906,  17.7969,  ..., -16.8125,  29.6406, -48.3125],
        [ 13.8125,  -6.4609,   0.5142,  ...,  48.0312,  25.1406, -27.5156],
        ...,
        [-54.4375,  -1.4375,  -0.4214,  ...,   5.4297,  17.2812, -39.0625],
        [  9.6094, -52.2500,  43.0000,  ..., -45.8125, -49.3125,  32.7188],
        [

### Profile the Non-Fusion MatMulBias Kernel

#### Using Triton Proton to generate the report

In [5]:
!python nonfusion_matmulbias.py --profile

Profiling the nonfusion matmulbias kernel ...
nonfusion_triton_output: tensor([[  2.7480,  43.6250,  30.9531,  ...,  26.2500,   8.8203,   6.4609],
        [  7.1562, -21.3906,  17.7969,  ..., -16.8125,  29.6562, -48.3125],
        [ 13.8125,  -6.4609,   0.5146,  ...,  48.0000,  25.1562, -27.5156],
        ...,
        [-54.4375,  -1.4375,  -0.4214,  ...,   5.4297,  17.2812, -39.0625],
        [  9.6094, -52.2500,  43.0000,  ..., -45.8438, -49.3125,  32.6875],
        [-12.8516,   9.5391,  53.3750,  ...,  -9.3438,   7.6797,  54.1250]],
       device='cuda:0', dtype=torch.float16)
Proton profile results for nonfusion_matmulbias
998.325 3.596 ROOT
├─ 868.157 nan _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
├─ 44.324 0.580 bias_kernel [M=1024, N=1024] [BM=128, BN=128]
├─ 47.207 0.540 bias_kernel [M=1024, N=1024] [BM=128, BN=256]
└─ 38.637 91.598 matmul_kernel [M=1024, N=1024, K=1024] [BM=128, BN=128 BK=64]

Legend (Metric: time/ms (inc) M

#### Using ROCm Systems Compute to generate the report

In [6]:
!ROCPROF=rocprofiler-sdk rocprof-compute profile -n nonfusion_matmulbias -- python nonfusion_matmulbias.py


                                 __                                       _
 _ __ ___   ___ _ __  _ __ ___  / _|       ___ ___  _ __ ___  _ __  _   _| |_ ___
| '__/ _ \ / __| '_ \| '__/ _ \| |_ _____ / __/ _ \| '_ ` _ \| '_ \| | | | __/ _ \
| | | (_) | (__| |_) | | | (_) |  _|_____| (_| (_) | | | | | | |_) | |_| | ||  __/
|_|  \___/ \___| .__/|_|  \___/|_|        \___\___/|_| |_| |_| .__/ \__,_|\__\___|
               |_|                                           |_|

WARNING DEPRECATION WARNING: rocm-smi is deprecated in ROCm 7.0 and will be removed from rocprof-compute in ROCm 7.1. Please migrate to amd-smi for compute partition parsing. For migration help, see https://github.com/ROCm/amdsmi
   INFO Rocprofiler-Compute version: 3.2.3
   INFO Profiler choice: rocprofiler-sdk
   INFO Path: /workspace/user/workloads/nonfusion_matmulbias/MI300X_A1
   INFO Target: MI300X_A1
   INFO Command: python nonfusion_matmulbias.py
   INFO Kernel Selection: None
   INFO Dispatch Selection: None
   

### Analyze the Non-Fusion MatMulBias Kernel

#### Using Triton Proton to analyze the report

In [7]:
!proton-viewer -m flops16,time/s -f full nonfusion_matmulbias.hatchet

3590238240768.000 0.998 ROOT
├─ nan 0.868 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
├─ 25706889216.000 0.044 bias_kernel [M=1024, N=1024] [BM=128, BN=128]
├─ 25478299648.000 0.047 bias_kernel [M=1024, N=1024] [BM=128, BN=256]
└─ 3539053051904.000 0.039 matmul_kernel [M=1024, N=1024, K=1024] [BM=128, BN=128 BK=64]

Legend (Metric: flops16 (inc) Min: 25478299648.00 Max: 3590238240768.00)
█ 3233762246656.00 - 3590238240768.00
█ 2520810258432.00 - 3233762246656.00
█ 1807858270208.00 - 2520810258432.00
█ 1094906281984.00 - 1807858270208.00
█ 381954293760.00 - 1094906281984.00
█ 25478299648.00 - 381954293760.00

name User code    ◀  Only in left graph    ▶  Only in right graph



#### Using ROCm Systems Compute to analyze the report

In [8]:
!rocprof-compute analyze -p workloads/nonfusion_matmulbias/MI300X_A1


                                 __                                       _
 _ __ ___   ___ _ __  _ __ ___  / _|       ___ ___  _ __ ___  _ __  _   _| |_ ___
| '__/ _ \ / __| '_ \| '__/ _ \| |_ _____ / __/ _ \| '_ ` _ \| '_ \| | | | __/ _ \
| | | (_) | (__| |_) | | | (_) |  _|_____| (_| (_) | | | | | | |_) | |_| | ||  __/
|_|  \___/ \___| .__/|_|  \___/|_|        \___\___/|_| |_| |_| .__/ \__,_|\__\___|
               |_|                                           |_|

   INFO Analysis mode = cli
   INFO [analysis] deriving rocprofiler-compute metrics...
WARNING PC sampling: can not detect pc sampling method without /workspace/user/workloads/nonfusion_matmulbias/MI300X_A1/ps_file_pc_sampling_host_trap.csv 

--------------------------------------------------------------------------------
0. Top Stats
0.1 Top Kernels
╒════╤══════════════════════════════════════════╤═════════╤═════════════╤════════════╤══════════════╤═══════╕
│    │ Kernel_Name                              │   Count │    

## Fusion MatMulBias Kernel

In [9]:
%%writefile  fusion_matmulbias.py
import argparse
import torch

import triton
import triton.language as tl
import triton.profiler as proton


def _matmul_launch_metadata(grid, kernel, args):
    ret = {}
    M, N, K, WS = args["M"], args["N"], args["K"], args.get("WARP_SPECIALIZE", False)
    BM, BN, BK = args["BLOCK_SIZE_M"], args["BLOCK_SIZE_N"], args["BLOCK_SIZE_K"]
    ws_str = "_ws" if WS else ""
    ret["name"] = f"{kernel.name}{ws_str} [M={M}, N={N}, K={K}] [BM={BM}, BN={BN}, BK={BK}]"
    if "output_ptr" in args:
        bytes_per_elem = args["output_ptr"].element_size()
    else:
        bytes_per_elem = 2
    ret[f"flops{bytes_per_elem * 8}"] = 2. * M * N * K
    ret["bytes"] = bytes_per_elem * (M * K + N * K + M * N)
    return ret

def matmul_autotune_config(pre_hook=None):
    return [
        triton.Config({'BLOCK_SIZE_M': BM, 'BLOCK_SIZE_N': BN, "BLOCK_SIZE_K": BK, "GROUP_SIZE_M": 8}, num_stages=s,
            num_warps=w, pre_hook=pre_hook)
        for BM in [128]
        for BN in [128, 256]
        for BK in [64, 128]
        for s in ([3, 4, 5])
        for w in [4, 8]
    ]

# MatMulBias Fusion kernel
@triton.autotune(
    configs=matmul_autotune_config(),
    key=['M', 'N', 'K'],
)
@triton.jit(launch_metadata=_matmul_launch_metadata)
def matmulbias_kernel(
        a_ptr, b_ptr, c_ptr,
        bias_ptr,
        M, N, K,
        stride_am, stride_ak,
        stride_bk, stride_bn,
        stride_cm, stride_cn,
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,
        GROUP_SIZE_M: tl.constexpr
):
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    tl.assume(pid_m >= 0)
    tl.assume(pid_n >= 0)
    tl.assume(stride_am > 0)
    tl.assume(stride_ak > 0)
    tl.assume(stride_bn > 0)
    tl.assume(stride_bk > 0)
    tl.assume(stride_cm > 0)
    tl.assume(stride_cn > 0)

    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        accumulator = tl.dot(a, b, accumulator)
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk
    c = accumulator.to(tl.float16)

    bias = tl.load(bias_ptr + offs_bn, mask=offs_bn < N, other=0.0)
    output = c + bias[None, :]

    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, output, mask=c_mask)


# MatMulBias kernel wrapper function
def matmulbiasfusion(a: torch.Tensor, b: torch.Tensor, bias: torch.Tensor):
    assert a.shape[1] == b.shape[0], "Incompatible dimensions"
    assert a.is_contiguous(), "Matrix A must be contiguous"
    M, K = a.shape
    K, N = b.shape
    assert bias.shape[0] == N, "BIAS has incompatible dimensions"
    o = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']), )
    matmulbias_kernel[grid](
        a, b, o,
        bias,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        o.stride(0), o.stride(1),
    )
    
    return o


def get_rtol():
    target = triton.runtime.driver.active.get_current_target()
    if target.backend == "hip":
        if target.arch == "gfx90a":
            return 1e-2
        elif target.arch == "gfx942":
            return 1e-2
    else:
        return 0

def show_profile(profile_name):
    import triton.profiler.viewer as proton_viewer
    metric_names = ["time/ms", "tflop16/s"]
    file_name = f"{profile_name}.hatchet"
    tree, metrics = proton_viewer.parse(metric_names, file_name)

    print(f"Proton profile results for {profile_name}")
    proton_viewer.print_tree(tree, metrics)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--profile", action="store_true")
    parser.add_argument("--verify", action="store_true")
    args = parser.parse_args()

    # Test matrices
    torch.manual_seed(0)
    M = 1024
    N = 1024
    a = torch.randn((M, N), device='cuda', dtype=torch.float16)
    b = torch.randn((N, M), device='cuda', dtype=torch.float16)
    bias = torch.randn(N, device='cuda', dtype=torch.float16)

    if args.profile:
        print("Profiling the fusion matmulbias kernel ...")
        proton.start("fusion_matmulbias", hook="triton")
    else:
        print("Running the fusion matmulbias kernel ...")
    fusion_triton_output = matmulbiasfusion(a, b, bias)
    if args.profile:
        proton.finalize()
    print(f"fusion_triton_output: {fusion_triton_output}")

    if args.profile:
        show_profile("fusion_matmulbias")

    if args.verify:
        torch_output = torch.addmm(bias, a, b)
        print(f"torch_output: {torch_output}")
        print("Verifying triton results with torch ...")
        triton.testing.assert_close(fusion_triton_output, torch_output, atol=1e-2, rtol=get_rtol())
        print("OK")


Overwriting fusion_matmulbias.py


### Pre-Build the Fusion MatMulBias Kernel

In [10]:
!python fusion_matmulbias.py --verify

Running the fusion matmulbias kernel ...
fusion_triton_output: tensor([[  2.7480,  43.6250,  30.9531,  ...,  26.2500,   8.8203,   6.4609],
        [  7.1562, -21.3906,  17.7969,  ..., -16.8125,  29.6562, -48.3125],
        [ 13.8125,  -6.4609,   0.5146,  ...,  48.0000,  25.1562, -27.5156],
        ...,
        [-54.4375,  -1.4375,  -0.4214,  ...,   5.4297,  17.2812, -39.0625],
        [  9.6094, -52.2500,  43.0000,  ..., -45.8438, -49.3125,  32.6875],
        [-12.8516,   9.5391,  53.3750,  ...,  -9.3438,   7.6797,  54.1250]],
       device='cuda:0', dtype=torch.float16)
torch_output: tensor([[  2.7480,  43.6562,  30.9531,  ...,  26.2500,   8.8203,   6.4609],
        [  7.1602, -21.3906,  17.7969,  ..., -16.8125,  29.6406, -48.3125],
        [ 13.8125,  -6.4609,   0.5142,  ...,  48.0312,  25.1406, -27.5156],
        ...,
        [-54.4375,  -1.4375,  -0.4214,  ...,   5.4297,  17.2812, -39.0625],
        [  9.6094, -52.2500,  43.0000,  ..., -45.8125, -49.3125,  32.7188],
        [-12.85

### Profile the Fusion MatMulBias Kernel

#### Using Triton Proton to generate the report

In [11]:
!python fusion_matmulbias.py --profile

Profiling the fusion matmulbias kernel ...
fusion_triton_output: tensor([[  2.7480,  43.6250,  30.9531,  ...,  26.2500,   8.8203,   6.4609],
        [  7.1562, -21.3906,  17.7969,  ..., -16.8125,  29.6562, -48.3125],
        [ 13.8125,  -6.4609,   0.5146,  ...,  48.0000,  25.1562, -27.5156],
        ...,
        [-54.4375,  -1.4375,  -0.4214,  ...,   5.4297,  17.2812, -39.0625],
        [  9.6094, -52.2500,  43.0000,  ..., -45.8438, -49.3125,  32.6875],
        [-12.8516,   9.5391,  53.3750,  ...,  -9.3438,   7.6797,  54.1250]],
       device='cuda:0', dtype=torch.float16)
Proton profile results for fusion_matmulbias
96.946 37.414 ROOT
├─ 56.534 nan _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
└─ 40.412 89.754 matmulbias_kernel [M=1024, N=1024, K=1024] [BM=128, BN=128, BK=64]

Legend (Metric: time/ms (inc) Min: 40.41 Max: 96.95)
█ 91.29 - 96.95
█ 79.99 - 91.29
█ 68.68 - 79.99
█ 57.37 - 68.68
█ 46.07 - 57.37
█ 40.41 - 46.07

name User 

#### Using ROCm Systems Compute to generate the report

In [12]:
!ROCPROF=rocprofiler-sdk rocprof-compute profile -n fusion_matmulbias -- python fusion_matmulbias.py


                                 __                                       _
 _ __ ___   ___ _ __  _ __ ___  / _|       ___ ___  _ __ ___  _ __  _   _| |_ ___
| '__/ _ \ / __| '_ \| '__/ _ \| |_ _____ / __/ _ \| '_ ` _ \| '_ \| | | | __/ _ \
| | | (_) | (__| |_) | | | (_) |  _|_____| (_| (_) | | | | | | |_) | |_| | ||  __/
|_|  \___/ \___| .__/|_|  \___/|_|        \___\___/|_| |_| |_| .__/ \__,_|\__\___|
               |_|                                           |_|

WARNING DEPRECATION WARNING: rocm-smi is deprecated in ROCm 7.0 and will be removed from rocprof-compute in ROCm 7.1. Please migrate to amd-smi for compute partition parsing. For migration help, see https://github.com/ROCm/amdsmi
   INFO Rocprofiler-Compute version: 3.2.3
   INFO Profiler choice: rocprofiler-sdk
   INFO Path: /workspace/user/workloads/fusion_matmulbias/MI300X_A1
   INFO Target: MI300X_A1
   INFO Command: python fusion_matmulbias.py
   INFO Kernel Selection: None
   INFO Dispatch Selection: None
   INFO H

### Analyze the Fusion MatMulBias Kernel

#### Using Triton Proton to analyze the report

In [13]:
!proton-viewer -m flops16,time/s -f full fusion_matmulbias.hatchet

3627099881472.000 0.097 ROOT
├─ nan 0.057 _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
└─ 3627099881472.000 0.040 matmulbias_kernel [M=1024, N=1024, K=1024] [BM=128, BN=128, BK=64]

Legend (Metric: flops16 (inc) Min: 3627099881472.00 Max: 3627099881472.00)
█ 3627099881472.00 - 3627099881472.00
█ 3627099881472.00 - 3627099881472.00
█ 3627099881472.00 - 3627099881472.00
█ 3627099881472.00 - 3627099881472.00
█ 3627099881472.00 - 3627099881472.00
█ 3627099881472.00 - 3627099881472.00

name User code    ◀  Only in left graph    ▶  Only in right graph



#### Using ROCm Systems Compute to analyze the report

In [14]:
!rocprof-compute analyze -p workloads/fusion_matmulbias/MI300X_A1


                                 __                                       _
 _ __ ___   ___ _ __  _ __ ___  / _|       ___ ___  _ __ ___  _ __  _   _| |_ ___
| '__/ _ \ / __| '_ \| '__/ _ \| |_ _____ / __/ _ \| '_ ` _ \| '_ \| | | | __/ _ \
| | | (_) | (__| |_) | | | (_) |  _|_____| (_| (_) | | | | | | |_) | |_| | ||  __/
|_|  \___/ \___| .__/|_|  \___/|_|        \___\___/|_| |_| |_| .__/ \__,_|\__\___|
               |_|                                           |_|

   INFO Analysis mode = cli
   INFO [analysis] deriving rocprofiler-compute metrics...
WARNING PC sampling: can not detect pc sampling method without /workspace/user/workloads/fusion_matmulbias/MI300X_A1/ps_file_pc_sampling_host_trap.csv 

--------------------------------------------------------------------------------
0. Top Stats
0.1 Top Kernels
╒════╤══════════════════════════════════════════╤═════════╤════════════╤════════════╤══════════════╤═══════╕
│    │ Kernel_Name                              │   Count │    Sum(